In [ ]:
from torch import nn, optim
import torch.nn.functional as F

class Classifier(nn.Module):
    def __init__(self, dropout_p):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, x):
        # make sure input tensor is flattened
        x = x.view(x.shape[0], -1)

        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.dropout(F.relu(self.fc3(x)))
        x = F.log_softmax(self.fc4(x), dim=1)

        return x


# class Classifier(nn.Module):
#     def __init__(self, dropout_p=0.5):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(784, 256),
#             nn.ReLU(),
#             nn.Dropout(dropout_p),
#             nn.Linear(256, 128),
#             nn.ReLU(),
#             nn.Dropout(dropout_p),
#             nn.Linear(128, 64),
#             nn.ReLU(),
#             nn.Linear(64, 10),
#             nn.LogSoftmax(dim=1)
#         )

#     def forward(self, x):
#         return self.net(x)


In [ ]:
import torch
from torch import nn, optim
from dataclasses import dataclass
import logging

# -------------------------
# Config
# -------------------------

@dataclass(frozen=True)
class Config:
    epochs: int = 30
    lr: float = 3e-3
    seed: int = 42
    grad_clip: float = 1.0


# -------------------------
# Utilities
# -------------------------

def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -------------------------
# Train / Eval functions
# -------------------------

def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        log_probs = model(images)
        loss = criterion(log_probs, labels)

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss detected")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            log_probs = model(images)
            loss = criterion(log_probs, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = log_probs.argmax(dim=1)
            correct += (preds == labels).sum().item()

    avg_loss = total_loss / total_samples
    accuracy = correct / total_samples

    return avg_loss, accuracy


# -------------------------
# Main training loop
# -------------------------

def train(model, trainloader, testloader, config: Config):
    set_seed(config.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.lr)

    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(config.epochs):
        train_loss = train_one_epoch(
            model,
            trainloader,
            criterion,
            optimizer,
            device,
            config.grad_clip,
        )

        val_loss, val_acc = evaluate(
            model,
            testloader,
            criterion,
            device,
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{config.epochs}] | ")
        print(f"Train Loss: {train_loss:.4f} | ")
        print(f"Val Loss: {val_loss:.4f} | ")
        print(f"Val Acc: {val_acc:.4f}")

    return train_losses, val_losses, val_accuracies


In [ ]:
config = Config(
    epochs=30,
    lr=3e-3,
    seed=42,
    grad_clip=1.0
)

model = Classifier()

train_losses, val_losses, val_accuracies = train(
    model,
    trainloader,
    testloader,
    config
)

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    input_dim: int = 784
    hidden_dims: tuple = (256, 128, 64)
    num_classes: int = 10
    dropout: float = 0.2


checkpoint = {
    "model_state": model.state_dict(),
    "model_config": model_config,
    "optimizer_state": optimizer.state_dict(),
    "epoch": epoch,
}
torch.save(checkpoint, "model.pt")

ckpt = torch.load("model.pt")
model = Classifier(ckpt["model_config"])
model.load_state_dict(ckpt["model_state"])

t = { 'te': 'tes', 2: 't'}